In [0]:
# ============================================
# Champion-Challenger Evaluation
# ============================================

import mlflow
import mlflow.pyfunc
import mlflow.sklearn

from mlflow import MlflowClient

# Registered model
MODEL_NAME = "oil_gas_demo.ml_model.random_forest_demand"

# Model aliases
CHAMPION_ALIAS = "champion"
CHALLENGER_ALIAS = "challenger"

# Validation dataset
VALIDATION_TABLE = "oil_gas_demo.ml_model.validation_data"

# MLflow client
client = MlflowClient()

print("Configuration loaded successfully")
print(f"Model: {MODEL_NAME}")
print(f"Champion alias: {CHAMPION_ALIAS}")
print(f"Challenger alias: {CHALLENGER_ALIAS}")
print(f"Validation table: {VALIDATION_TABLE}")

In [0]:
# ============================================
# Step 2: Retrieve model versions
# ============================================

# Get the registered model versions using aliases
champion_version = client.get_model_version_by_alias(
    MODEL_NAME, CHAMPION_ALIAS
)

challenger_version = client.get_model_version_by_alias(
    MODEL_NAME, CHALLENGER_ALIAS
)

# Extract version and run IDs
champion_version_number = champion_version.version
champion_run_id = champion_version.run_id

challenger_version_number = challenger_version.version
challenger_run_id = challenger_version.run_id

# Display details
print("========== CHAMPION ==========")
print("Version:", champion_version_number)
print("Run ID:", champion_run_id)

print("\n========== CHALLENGER ==========")
print("Version:", challenger_version_number)
print("Run ID:", challenger_run_id)

In [0]:
# ============================================
# Step 3: Load model-specific encoders
# ============================================

import joblib
import mlflow

# Download Champion encoder
champion_encoder_path = mlflow.artifacts.download_artifacts(
    run_id=champion_run_id,
    artifact_path="encoder/encoder.joblib"
)

champion_encoder = joblib.load(champion_encoder_path)

print("Champion encoder loaded successfully!")


# Download Challenger encoder
challenger_encoder_path = mlflow.artifacts.download_artifacts(
    run_id=challenger_run_id,
    artifact_path="encoder/encoder.joblib"
)

challenger_encoder = joblib.load(challenger_encoder_path)

print("Challenger encoder loaded successfully!")

In [0]:
# ============================================
# Step 4: Load validation data
# ============================================

VALIDATION_TABLE = "oil_gas_demo.ml_model.validation_data"

validation_df = spark.table(VALIDATION_TABLE)

print("Validation rows:", validation_df.count())
print("Validation columns:", validation_df.columns)

display(validation_df.limit(10))

In [0]:
# ============================================
# Step 5: Generate predictions
# ============================================

import pandas as pd

# Load models from MLflow
champion_model = mlflow.sklearn.load_model(f"runs:/{champion_run_id}/random_forest_demand_model")
challenger_model = mlflow.sklearn.load_model(f"runs:/{challenger_run_id}/random_forest_demand_model")

# Prepare validation features (same preprocessing as training)
val_pd = validation_df.toPandas()

# Handle missing values in time-series feature columns
for col_name in ["demand_lag_1", "demand_lag_7", "rolling_avg_7"]:
    if col_name in val_pd.columns:
        val_pd = val_pd.fillna({col_name: 0.0})

# Categorical columns and columns to drop (same as training)
cat_cols = ["product_name", "destination_city"]
DROP_COLS = ["feature_uid", "transaction_date", "total_demand", "product_name", "destination_city"]

drop_present = [c for c in DROP_COLS if c in val_pd.columns]

# Prepare champion features
champion_encoded = champion_encoder.transform(val_pd[cat_cols])
champion_encoded_cols = champion_encoder.get_feature_names_out(cat_cols).tolist()
champion_encoded_df = pd.DataFrame(champion_encoded, columns=champion_encoded_cols, index=val_pd.index)
champion_features = pd.concat([val_pd.drop(columns=drop_present), champion_encoded_df], axis=1)

# Prepare challenger features
challenger_encoded = challenger_encoder.transform(val_pd[cat_cols])
challenger_encoded_cols = challenger_encoder.get_feature_names_out(cat_cols).tolist()
challenger_encoded_df = pd.DataFrame(challenger_encoded, columns=challenger_encoded_cols, index=val_pd.index)
challenger_features = pd.concat([val_pd.drop(columns=drop_present), challenger_encoded_df], axis=1)

# Predict with each model
champion_predictions = champion_model.predict(
    champion_features
)

challenger_predictions = challenger_model.predict(
    challenger_features
)

print("Champion predictions generated:", len(champion_predictions))
print("Challenger predictions generated:", len(challenger_predictions))

In [0]:
# Compare model performance

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_true = validation_df.select("total_demand").toPandas()["total_demand"]

def evaluate(name, predictions):
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_true, predictions)),
        "R2": r2_score(y_true, predictions)
    }

comparison_df = pd.DataFrame([
    evaluate("Champion", champion_predictions),
    evaluate("Challenger", challenger_predictions)
])

display(comparison_df)

In [0]:
# Step 7: Promote Challenger if approved

# Determine promotion decision based on validation MAE
champion_mae = comparison_df.loc[comparison_df["Model"] == "Champion", "MAE"].values[0]
challenger_mae = comparison_df.loc[comparison_df["Model"] == "Challenger", "MAE"].values[0]

promotion_result = {
    "decision": "PROMOTE" if challenger_mae < champion_mae else "KEEP_CHAMPION",
    "champion_mae": champion_mae,
    "challenger_mae": challenger_mae
}

print(f"Champion MAE: {champion_mae:.2f} | Challenger MAE: {challenger_mae:.2f}")
print(f"Decision: {promotion_result['decision']}")

if promotion_result["decision"] == "PROMOTE":

    # Keep track of the current Champion
    old_champion = client.get_model_version_by_alias(
        MODEL_NAME, "champion"
    )

    # Preserve the previous Champion version
    client.set_registered_model_alias(
        MODEL_NAME,
        "previous-champion",
        old_champion.version
    )

    # Promote Challenger
    client.set_registered_model_alias(
        MODEL_NAME,
        "champion",
        challenger_version_number
    )

    print(f"Version {challenger_version_number} promoted to Champion!")

else:
    print("Challenger not promoted.")
    print(f"Champion remains Version {champion_version_number}.")

In [0]:
# Step 8: Load current Champion model

import mlflow
import mlflow.sklearn

MODEL_NAME = "oil_gas_demo.ml_model.random_forest_demand"
CHAMPION_MODEL_URI = f"models:/{MODEL_NAME}@champion"

champion_model = mlflow.sklearn.load_model(
    CHAMPION_MODEL_URI
)

print("Champion model loaded successfully!")